# 03 — Compression analysis

Per-arm inspection, before any joint-gain claim is made. The point of this notebook is to catch
the failure modes that produce excellent-looking results:

1. **Measured sparsity below target** — masks were not applied, or the optimiser refilled pruned
   positions.
2. **`is_converted` false** — the model is numerically quantised but still FP32 on disk, so its
   size and latency mean nothing.
3. **Low `storage_efficiency`** — the artefact is far larger than its budget allows.
4. **Unmatched optimiser steps** — the joint arm trained longer, so any gain is confounded.
5. **Near-zero `sparsity_realisation`** — expected for unstructured sparsity, and a finding to
   report rather than a bug to fix.

In [ ]:
from scale_aware_compression.metrics.compression import effective_compression_ratio
from scale_aware_compression.metrics.efficiency import theoretical_speedup_from_sparsity
from scale_aware_compression.visualisation.tables import rows_to_markdown

# What each budget implies, before any measurement.
budgets = [("moderate", 0.5, 8), ("aggressive", 0.7, 4)]
rows = [
    {
        "budget": label,
        "sparsity": sparsity,
        "bits": bits,
        "size_reduction": effective_compression_ratio(1000, int(1000 * sparsity), bits),
        "latency_bound": theoretical_speedup_from_sparsity(sparsity),
    }
    for label, sparsity, bits in budgets
]
print(rows_to_markdown(rows))

In [ ]:
from scale_aware_compression.compression.schedules import schedule_values

# Both arms share these, so a divergence would silently change the comparison.
for schedule in ("linear", "cubic"):
    points = schedule_values(
        schedule=schedule, final_sparsity=0.7, initial_sparsity=0.0, end_step=500, num_points=6
    )
    formatted = ", ".join(f"{step}:{value:.3f}" for step, value in points)
    print(f"{schedule:7s} {formatted}")

In [ ]:
from scale_aware_compression.experiments.runner import ExperimentTracker

records = ExperimentTracker("../outputs/metrics").load_all()

issues = []
for record in records:
    statistics = record.get("compression", {}).get("statistics", {})
    if not statistics:
        continue
    identifier = record["experiment_id"]
    target = statistics.get("target_sparsity") or 0.0
    measured = (statistics.get("measured_sparsity_percentage") or 0.0) / 100
    if target > 0 and abs(measured - target) > 0.02:
        issues.append(f"{identifier}: sparsity {measured:.3f} against a target of {target:.3f}")
    if statistics.get("target_bits", 32) < 32 and not statistics.get("is_converted", False):
        issues.append(f"{identifier}: quantised but never converted")

print("\n".join(issues) if issues else f"No issues found across {len(records)} record(s).")

In [ ]:
from scale_aware_compression.metrics.efficiency import training_cost_overhead

# The fairness check. Anything other than 1.00 means the arms were not budget-matched.
by_key = {}
for record in records:
    method = record.get("compression_method")
    if method not in {"sequential", "joint"}:
        continue
    key = (record["model_name"], record.get("budget_label"), record.get("seed"))
    steps = record.get("compression", {}).get("total_optimiser_steps", 0)
    by_key.setdefault(key, {})[method] = steps

for key, steps in sorted(by_key.items()):
    if "sequential" in steps and "joint" in steps and steps["sequential"]:
        overhead = training_cost_overhead(steps["joint"], steps["sequential"])
        flag = "ok" if abs(overhead - 1.0) < 0.01 else "MISMATCH"
        print(f"{key}: {overhead:.2f}x  {flag}")
    else:
        print(f"{key}: incomplete pair {sorted(steps)}")

## Still to do

- per-layer sparsity heatmaps, to see whether global ranking left some layers untouched
- weight-distribution histograms before and after quantisation, per arm
- read the generation samples at the aggressive budget and check for degenerate repetition